# University Campus Routing Problem 

In [65]:
import heapq # We use this module because it implements priority queue
import math # For Euclidian distance
import itertools

### Define classes that the system will use 

In [105]:
class Node: 
    def __init__(self, coordenate, parent = None, path_cost = 0, floor = 1, building = "...", action = None):
        self.coordenate = coordenate
        self.parent = parent 
        self.path_cost = path_cost
        self.floor = floor
        self.building = building
        self.action = action
        # We inicialize the node for root

class Problem: 
    def __init__(self, start, goal):
        self.start = start
        self.goal = goal 
        

### Moves availaible on the problem while Walking

In [106]:
movesWhenWalking = {
    'Down': (0, 1),
    'Up': (0, -1),
    'Right': (1, 0),
    'Left': (-1, 0)
}

### Define actions availaible 


In [107]:
def actionsWhenWalking(node, grid):
    x, y = node.coordenate
    available_actions = {}

    for direction, (dx, dy) in movesWhenWalking.items():
        newX = x + dx
        newY = y + dy

        # We check if the new position is on the map
        isInsideMap = (0 <= newY < len(grid)) and (0 <= newX < len(grid[0]))

        if isInsideMap:
            if grid[newY][newX] != "#":
                available_actions[direction] = True # Aquí hay algo
            else:
                available_actions[direction] = False
        else:
            available_actions[direction] = False 

    return available_actions


### Related with Heuristic Function 

In [108]:
def euclidean_distance(a, b):
    
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)

def heuristicFunction(problem, node): 
    return euclidean_distance(node.coordenate,problem.goal["coordenate"])

### Goal State

In [109]:
def is_partial_goal(problem, node): 
    if heuristicFunction(problem, node) == 0: 
        return True 
    return False

def is_full_goal(problem, node): 
    if node.floor == problem.goal.floor: 
        return True 
    return False 

### Expand Nodes

In [110]:
def expand(problem, node, grid):
    
    available_actions = actionsWhenWalking(node, grid)

    for direction, can_move in available_actions.items():
        if can_move:
            x, y = node.coordenate
            dx, dy = movesWhenWalking[direction]
            new_coord = (x + dx, y + dy)
            new_node = Node(
                coordenate=new_coord,
                parent=node,
                path_cost=node.path_cost + 1,  # Each plane movement has 1 unit cost
                floor=node.floor,
                building=node.building,
                action=direction
            )
            yield new_node


### A* search. 
In this case we mixed Breadth-First-Search with Greedy Search. Due to each plane movement has 1 unit value.

In [ ]:
def a_star(problem, grid):
    start_node = Node(coordenate=problem.start["coordenate"], floor=problem.start["piso"])
    frontier = []
    counter = itertools.count()  
    heapq.heappush(frontier, (heuristicFunction(problem, start_node), next(counter), start_node))
    reached = {start_node.coordenate: 0}
    
    while frontier:
        _, _, node = heapq.heappop(frontier)
        
        if node.coordenate == problem.goal["coordenate"]:
            return node
        
        for child in expand(problem, node, grid):
            g = child.path_cost
            if child.coordenate not in reached or g < reached[child.coordenate]:
                reached[child.coordenate] = g
                f = g + heuristicFunction(problem, child)
                heapq.heappush(frontier, (f, next(counter), child))
    
    return None


### Reconstruct Path

In [112]:
def reconstruct_path(node):
    
    path = []
    states = []
    current = node
    
    while current.parent is not None:  # Go until initial state
        path.append(current.action)
        states.append(current)
        current = current.parent

    states.append(current)  
    path.reverse()          
    states.reverse()        
    
    return path, states


### Define problem 

In [125]:
problem = Problem(
    start={"coordenate": (6,15), "piso":1, "edificio":"Pasillo"},
    goal={"coordenate": (16,6), "piso":1, "edificio":"B38"},
)

grid = [
    [" ", "#", " ",  " ", "#",  " ",  " ", "#", " ",  " ", "#", " ",  " ", "#", " ",  " ", " "],
    [" ", "#", " ",  " ", "#",  " ",  " ", "#", " ",  " ", "#", " ",  " ", "#", " ",  " ", " "],
    [" ", " ", "B30"," ", " ", "B31"," ", " ", "B32"," ", " ", "B33"," ", " ", "B34"," ", " "],
    [" ", "#", " ",  "#", "#", " ",  "#", " ", "#",  "#", " ", "#",  " ", "#", " ",  "#", " "],
    [" ", "#", " ",  " ", " ", " ",  "#", " ", "#",  " ", " ", "#",  " ", "#", " ",  "#", " "],
    [" ", " ", " ",  "#", " ", " ",  " ", " ", " ",  "#", " ", " ",  " ", " ", " ",  "#", " "],
    [" ", "#", " ",  " ", "B35"," ",  "#", " ", "B36"," ", "#", " ",  "B37"," ", "#", " ", "B38"],
    [" ", "#", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", " "],
    [" ", " ", " ",  " ", "#", " ",  " ", " ", "#",  " ", " ", " ",  "#", " ", " ",  " ", " "],
    ["#", "#", " ",  "#", "#", " ",  "#", " ", " ",  " ", "#", " ",  "#", " ", "#",  "#", " "],
    [" ", " ", " ",  " ", "#", " ",  "C", " ", "#",  " ", "#", " ",  " ", " ", "B",  " ", " "],
    [" ", "#", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", " "],
    [" ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " "],
    [" ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " "],
    [" ", "#", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  "#", " "],
    [" ", " ", " ",  "#", "#", "#",  " ", "#", "#",  "#", " ", "#",  "#", "#", " ",  " ", " "],
]

### Solve the problem 

In [ ]:
result_node = a_star(problem, grid)

# Reconstruct path and states
ruta, estados = reconstruct_path(result_node)

print("Ruta:", ruta)
print("Costo total:", result_node.path_cost)


Ruta: ['Up', 'Right', 'Right', 'Up', 'Up', 'Right', 'Right', 'Right', 'Up', 'Up', 'Right', 'Right', 'Up', 'Up', 'Right', 'Right', 'Up', 'Up', 'Right']
Costo total: 19
